# Copula-GARCH

In [127]:
import numpy as np
import pandas as pd

from arch import arch_model 

from copulae import pseudo_obs # function to get pseudo-observations for copula fitting
from copulae import GaussianCopula

from scipy.stats import t # t-distribution for back-transformation of simulated pseudo-observations

import yfinance as yf

## Functions

Fitting GARCH models

In [ ]:
def fit_garch(returns, scale=100):
    """
    Fit a ARMA(1,1)-GARCH(1,1) model to the returns and return the fitted model.
    """
    model = arch_model(returns*scale, mean="AR", lags=1, vol="Garch", p=1, q=1, dist="t") # scale returns for better convergence

    res = model.fit(disp="off", show_warning=False) # disp="off" to suppress output, show_warning=False to ignore convergence warnings

    return res

## 1. Basics

### 1.1 Load data

In [3]:
TICKERS = ["NVDA", "AAPL", "WMT", "LLY", "JPM"]

prices = yf.download(TICKERS, start="2020-01-01", end="2024-06-30")["Close"]

returns = np.log(prices / prices.shift(1)).dropna() # log returns

returns.head()

[*********************100%***********************]  5 of 5 completed


Ticker,AAPL,JPM,LLY,NVDA,WMT
Date,,,,,
2020-01-03,-0.009769,-0.013285,-0.003334,-0.016135,-0.008867
2020-01-06,0.007936,-0.000795,0.003712,0.004185,-0.002038
2020-01-07,-0.004714,-0.017147,0.001888,0.012034,-0.009308
2020-01-08,0.015959,0.007771,0.009015,0.001874,-0.003438
2020-01-09,0.021018,0.003645,0.016393,0.010923,0.010278


### 1.2 Configuration

In [ ]:
WINDOW = 250
FORECAST = 100
N_SIM = 10000

ALPHA = 0.05

SCALE = 100 # scale factor for returns to improve GARCH convergence

SEED = 42

weights = np.ones(len(TICKERS)) / len(TICKERS)

## 2. Rolling window

# SANDBOX

### Understanding of GARCH spec

In [ ]:
def fit_garch(returns, scale=100):
    """
    Fit a ARMA(1,1)-GARCH(1,1) model to the returns and return the fitted model.
    """
    model = arch_model(returns*scale, mean="AR", lags=1, vol="Garch", p=1, q=1, dist="t") # scale returns for better convergence

    res = model.fit(disp="off", show_warning=False) # disp="off" to suppress output, show_warning=False to ignore convergence warnings

    return res

GARCH with mean=AR(1) 

In [178]:
res_nvda = fit_garch(returns["NVDA"], scale=SCALE)
res_nvda.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                              AR - GARCH Model Results                              
====================================================================================
Dep. Variable:                         NVDA   R-squared:                       0.004
Mean Model:                              AR   Adj. R-squared:                  0.003
Vol Model:                            GARCH   Log-Likelihood:               -2883.00
Distribution:      Standardized Student's t   AIC:                           5778.01
Method:                  Maximum Likelihood   BIC:                           5808.18
                                              No. Observations:                 1128
Date:                      Wed, May 13 2026   Df Residuals:                     1126
Time:                              14:06:58   Df Model:                            2
                                  Mean Model                                 
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
Const          0.3283  8.371e-02      3.922  8.787e-05      [  0.164,  0.492]
NVDA[1]       -0.0262  3.013e-02     -0.871      0.384 [-8.531e-02,3.282e-02]
                             Volatility Model                             
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega          0.4278      0.167      2.560  1.047e-02   [  0.100,  0.755]
alpha[1]       0.0921  2.413e-02      3.818  1.346e-04 [4.483e-02,  0.139]
beta[1]        0.8736  3.063e-02     28.524 5.976e-179   [  0.814,  0.934]
                              Distribution                              
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
nu             6.6004      1.435      4.599  4.251e-06 [  3.787,  9.413]
========================================================================

Covariance estimator: robust
"""

In [180]:
res_nvda.params

Const       0.328279
NVDA[1]    -0.026244
omega       0.427803
alpha[1]    0.092126
beta[1]     0.873582
nu          6.600357
Name: params, dtype: float64

In [182]:
# Mittelwert extrahieren
res_nvda.params["Const"]

np.float64(0.328278643304416)

In [184]:
res_nvda.resid[:10]

Date
2020-01-03         NaN
2020-01-06    0.047891
2020-01-07    0.886063
2020-01-08   -0.109317
2020-01-09    0.768913
2020-01-10    0.233907
2020-01-13    2.772818
2020-01-14   -2.130101
2020-01-15   -1.071612
2020-01-16    1.012465
Name: resid, dtype: float64

In [185]:
res_nvda.std_resid[:10]

Date
2020-01-03         NaN
2020-01-06    0.015767
2020-01-07    0.304131
2020-01-08   -0.038856
2020-01-09    0.283745
2020-01-10    0.089064
2020-01-13    1.091099
2020-01-14   -0.818186
2020-01-15   -0.411949
2020-01-16    0.398812
Name: std_resid, dtype: float64

In [186]:
res_nvda.resid[:10]/res_nvda.conditional_volatility[:10]

Date
2020-01-03         NaN
2020-01-06    0.015767
2020-01-07    0.304131
2020-01-08   -0.038856
2020-01-09    0.283745
2020-01-10    0.089064
2020-01-13    1.091099
2020-01-14   -0.818186
2020-01-15   -0.411949
2020-01-16    0.398812
dtype: float64

GARCH with mean=None

In [187]:
model_nvda2 = arch_model(returns["NVDA"]*100, mean="Zero", vol="Garch", p=1, q=1, dist="t") # scale returns for better convergence
res_nvda2 = model_nvda2.fit(disp="off", show_warning=False) # disp="off" to suppress output, show_warning=False to ignore convergence warnings
res_nvda2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                          Zero Mean - GARCH Model Results                           
====================================================================================
Dep. Variable:                         NVDA   R-squared:                       0.000
Mean Model:                       Zero Mean   Adj. R-squared:                  0.001
Vol Model:                            GARCH   Log-Likelihood:               -2892.58
Distribution:      Standardized Student's t   AIC:                           5793.16
Method:                  Maximum Likelihood   BIC:                           5813.28
                                              No. Observations:                 1129
Date:                      Wed, May 13 2026   Df Residuals:                     1129
Time:                              14:17:44   Df Model:                            0
                             Volatility Model                             
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega          0.4377      0.161      2.713  6.676e-03   [  0.121,  0.754]
alpha[1]       0.0898  2.344e-02      3.830  1.282e-04 [4.383e-02,  0.136]
beta[1]        0.8749  2.914e-02     30.026 4.550e-198   [  0.818,  0.932]
                              Distribution                              
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
nu             6.7466      1.502      4.491  7.102e-06 [  3.802,  9.691]
========================================================================

Covariance estimator: robust
"""

In [188]:
res_nvda2.params

omega       0.437701
alpha[1]    0.089767
beta[1]     0.874851
nu          6.746551
Name: params, dtype: float64

In [189]:
res_nvda2.resid[:10]

Date
2020-01-03   -1.613537
2020-01-06    0.418515
2020-01-07    1.203359
2020-01-08    0.187381
2020-01-09    1.092274
2020-01-10    0.533520
2020-01-13    3.087095
2020-01-14   -1.882840
2020-01-15   -0.693920
2020-01-16    1.358955
Name: resid, dtype: float64

### Fitting GARCH model

In [ ]:
res_aapl = fit_garch(returns["AAPL"])
res_jpm = fit_garch(returns["JPM"])

In [123]:
# Degree of freedom for the t-distribution of the standardized residuals
nu_aapl = res_aapl.params["nu"]
nu_jpm = res_jpm.params["nu"]

nu_aapl, nu_jpm

(np.float64(5.711445779124602), np.float64(5.695319945549439))

In [151]:
# Standardized residuals
residuals_std_aapl = res_aapl.std_resid 
residuals_std_jpm = res_jpm.std_resid

# Combine standardized residuals into a matrix for copula fitting
matrix_residuals_std = np.column_stack((residuals_std_aapl, residuals_std_jpm)) 
matrix_residuals_std

array([[-0.46026842, -0.57458822],
       [ 0.29209707, -0.08032252],
       [-0.26810081, -0.81802016],
       ...,
       [ 0.96435926, -0.33559904],
       [ 0.14365503,  0.61189905],
       [-0.95152938,  1.16462483]], shape=(1129, 2))

In [152]:
matrix_residuals_std2 = pd.DataFrame({"AAPL": residuals_std_aapl, "JPM": residuals_std_jpm})
matrix_residuals_std2

,AAPL,JPM
Date,,
2020-01-03,-0.460268,-0.574588
2020-01-06,0.292097,-0.080323
2020-01-07,-0.268101,-0.818020
2020-01-08,0.694984,0.304348
2020-01-09,0.953849,0.122560
...,...,...
2024-06-24,0.091446,0.883109
2024-06-25,0.161697,-0.385154
2024-06-26,0.964359,-0.335599


In [153]:
# Forecast values for the next day (horizon=1)
mu_aapl = res_aapl.forecast(horizon=1).mean.iloc[-1, 0] / 100 # scale back the mean to the original return scale
sigma_aapl = res_aapl.forecast(horizon=1).variance.iloc[-1, 0] ** 0.5  / 100 # ** 0.5 to get the standard deviation from the variance; scale back to the original return scale

mu_jpm = res_jpm.forecast(horizon=1).mean.iloc[-1, 0] / 100
sigma_jpm = res_jpm.forecast(horizon=1).variance.iloc[-1, 0] ** 0.5 / 100

mu_aapl, sigma_aapl

(np.float64(0.0012113870362165967), np.float64(0.018481205829487826))

### Fitting copula - Copulae Example

----------- Copulae example -----------

In [137]:
from copulae.datasets import load_residuals

residuals = load_residuals()
residuals.head()

,A,B,C,D,E,F,G
0,0.730967,0.530860,0.287320,1.193049,0.019040,1.100507,0.278214
1,2.067853,-1.181313,-2.546173,0.381538,-0.038734,0.269874,-0.603940
2,-2.181835,0.380326,0.928632,-0.316861,0.106473,-0.324854,-0.447824
3,0.445040,0.734531,-0.133299,-0.374091,0.173616,-0.319402,-0.775106
4,0.296363,3.024053,0.815791,1.168521,0.134044,1.110424,1.705190


In [96]:
residuals.shape

(394, 7)

In [97]:
matrix_residuals_std.shape

(1129, 2)

In [142]:
nodim = matrix_residuals_std.shape[1]
nodim

2

----------- Copulae example -----------

### Fitting copula

In [173]:
# Pseudo observations for copula fitting
u = pseudo_obs(matrix_residuals_std)
u

array([[0.29557522, 0.26283186],
       [0.64070796, 0.4840708 ],
       [0.3840708 , 0.18318584],
       ...,
       [0.85309735, 0.35752212],
       [0.57433628, 0.76725664],
       [0.13982301, 0.91238938]], shape=(1129, 2))

In [155]:
# Copula fitting
copula_gaussian = GaussianCopula(dim=matrix_residuals_std.shape[1]) # Initialize a Gaussian copula

copula_gauss_fit = copula_gaussian.fit(u) # Fit the copula to the pseudo-observations

In [172]:
# Sampling: Simulate from the fitted copula; values are in the unit square [0,1] because they are pseudo-observations
u_sim = copula_gauss_fit.random(N_SIM)  

u_sim

array([[0.19261216, 0.39551055],
       [0.5860545 , 0.4522988 ],
       [0.1849076 , 0.14576811],
       ...,
       [0.95768703, 0.48746703],
       [0.34137884, 0.36865164],
       [0.31954628, 0.54428091]], shape=(10000, 2))

In [ ]:
# Back-transform the simulated pseudo-observations to the original scale using the inverse CDF of the standardized residuals
# Uniform -> t-verteilte Residuen
z_sim_aapl = t.ppf(u_sim[:, 0], df=nu_aapl) 
z_sim_jpm = t.ppf(u_sim[:, 1], df=nu_jpm)

z_sim_aapl, z_sim_jpm

(array([ 2.96479562,  1.27909427,  1.16868669, ...,  0.16978892,
        -0.69599152, -0.32938989], shape=(10000,)),
 array([1.06753536, 0.04944186, 1.54375717, ..., 2.0754211 , 0.0881333 ,
        1.85348216], shape=(10000,)))

### Computation of PF return

In [ ]:
# Simulated returns
r_sim_aapl = mu_aapl + sigma_aapl * z_sim_aapl
r_sim_jpm = mu_jpm + sigma_jpm * z_sim_jpm
r_sim_aapl, r_sim_jpm

(array([ 0.05600439,  0.02485059,  0.02281013, ...,  0.00434929,
        -0.01165138, -0.00487614], shape=(10000,)),
 array([0.01473818, 0.00175687, 0.02081029, ..., 0.02758932, 0.00225021,
        0.02475947], shape=(10000,)))

In [165]:
r_sim = np.column_stack((r_sim_aapl, r_sim_jpm))
r_sim

array([[ 0.05600439,  0.01473818],
       [ 0.02485059,  0.00175687],
       [ 0.02281013,  0.02081029],
       ...,
       [ 0.00434929,  0.02758932],
       [-0.01165138,  0.00225021],
       [-0.00487614,  0.02475947]], shape=(10000, 2))

In [166]:
w = np.ones(matrix_residuals_std.shape[1]) / matrix_residuals_std.shape[1] # Equal weights for the portfolio
w

array([0.5, 0.5])

In [ ]:
np.ones(len(TICKERS)) / len(TICKERS)

In [168]:
# Simulated portfolio returns
r_sim_portfolio = w[0] * r_sim_aapl + w[1] * r_sim_jpm
r_sim_portfolio

array([ 0.03537128,  0.01330373,  0.02181021, ...,  0.01596931,
       -0.00470058,  0.00994167], shape=(10000,))

In [167]:
r_sim_pf = r_sim @ w
r_sim_pf

array([ 0.03537128,  0.01330373,  0.02181021, ...,  0.01596931,
       -0.00470058,  0.00994167], shape=(10000,))

### Risk forecasts

In [169]:
# Risk measure
VaR_95 = np.percentile(r_sim_pf, ALPHA)
VaR_95

np.float64(-0.0767792518775959)

## SANDBOX with R implementation

### Load return data

In [23]:
returns_aapl = returns["AAPL"]
returns_jpm = returns["JPM"]

### Using R's rugarch package in python

In [6]:
# rpy2 
import rpy2

import rpy2.robjects as ro
from rpy2.robjects.packages import importr # importr is used to import R packages


In [13]:
R --version

NameError: name 'R' is not defined

In [10]:
# loading R packages
utils = importr('utils') # utils is used to install R packages

copula = importr("copula")
rugarch = importr("rugarch")

In [11]:
# check versions of R packages
rugarch.__version__

'1.5-5'

In [12]:
# check versions of R packages
copula.__version__

'1.1-7'

#### Fitting the GARCH model

In [14]:
# GARCH(1,1) specification in R
spec = rugarch.ugarchspec(variance_model = ro.ListVector({'model': "sGARCH", 'garchOrder': ro.IntVector([1, 1])}), # specify GARCH(1,1) model
                          mean_model = ro.ListVector({'armaOrder': ro.IntVector([1, 1]), 'include.mean': True}), # specify ARMA(1,1) model for the mean
                          distribution_model = "std") # specify Student's t distribution for the innovations

In [24]:
# fit the model to the returns (scale returns for better convergence)
fit_aapl = rugarch.ugarchfit(spec, ro.FloatVector(returns_aapl.values * 100)) # fit the model to the returns (scale returns for better convergence)
fit_jpm = rugarch.ugarchfit(spec, ro.FloatVector(returns_jpm.values * 100)) # fit the model to the returns (scale returns for better convergence)

#### Extracting standardized residuals

In [25]:
residuals_aapl = np.array(rugarch.residuals(fit_aapl)) # Get the residuals from the fitted model
sigma_aapl = np.array(rugarch.sigma(fit_aapl)) # Get the conditional volatility from the fitted model
residuals_standardized_aapl = residuals_aapl / sigma_aapl # Standardize the residuals

residuals_jpm = np.array(rugarch.residuals(fit_jpm))
sigma_jpm = np.array(rugarch.sigma(fit_jpm))
residuals_standardized_jpm = residuals_jpm / sigma_jpm

#### Forecast values

In [26]:
# One-step-ahead forecast for mean and conditional volatility
forecast_aapl = rugarch.ugarchforecast(fit_aapl, n_ahead=1)
forecast_jpm = rugarch.ugarchforecast(fit_jpm, n_ahead=1)

In [27]:
# Mean forecast for the next day (scale back the mean forecast)
mu_forecast_aapl = np.array(rugarch.fitted(forecast_aapl))[0] / 100 # scale back the mean forecast
mu_forecast_jpm = np.array(rugarch.fitted(forecast_jpm))[0] / 100 # scale back the mean forecast

# Volatility forecast for the next day (scale back the volatility forecast)
sigma_forecast_aapl = np.array(rugarch.sigma(forecast_aapl))[0] / 100 # scale back the volatility forecast
sigma_forecast_jpm = np.array(rugarch.sigma(forecast_jpm))[0] / 100 # scale back the volatility forecast

In [29]:
mu_forecast_aapl, sigma_forecast_aapl, mu_forecast_jpm, sigma_forecast_jpm

(array([0.00147206]),
 array([0.01847455]),
 array([0.00103727]),
 array([0.01276176]))

In [35]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

matrix = np.array([a, b])
matrix

array([[1, 2, 3],
       [4, 5, 6]])

#### Pseudo observations

In [40]:
residuals_standardized_aapl

array([[-0.5360317 ],
       [ 0.32408252],
       [-0.30942911],
       ...,
       [ 0.95810541],
       [ 0.16134835],
       [-0.93760526]], shape=(1129, 1))

In [ ]:
# Speichern der standardisierten Residuen in der Matrix
residuals_standardized_matrix = np.column_stack((residuals_standardized_aapl, residuals_standardized_jpm))
residuals_standardized_matrix2 = np.array((residuals_standardized_aapl, residuals_standardized_jpm)) # rausnhemen
residuals_standardized_matrix, residuals_standardized_matrix2

(array([[-0.5360317 , -0.69340181],
        [ 0.32408252, -0.10065144],
        [-0.30942911, -0.96328314],
        ...,
        [ 0.95810541, -0.34285771],
        [ 0.16134835,  0.61168582],
        [-0.93760526,  1.1652201 ]], shape=(1129, 2)),
 array([[[-0.5360317 ],
         [ 0.32408252],
         [-0.30942911],
         ...,
         [ 0.95810541],
         [ 0.16134835],
         [-0.93760526]],
 
        [[-0.69340181],
         [-0.10065144],
         [-0.96328314],
         ...,
         [-0.34285771],
         [ 0.61168582],
         [ 1.1652201 ]]], shape=(2, 1129, 1)))

In [51]:
u2 = pseudo_obs(residuals_standardized_matrix) # Get the pseudo-observations from the standardized residuals
u2

array([[0.27168142, 0.22212389],
       [0.65486726, 0.47079646],
       [0.36548673, 0.1460177 ],
       ...,
       [0.85221239, 0.35309735],
       [0.58230088, 0.76725664],
       [0.14159292, 0.91150442]], shape=(1129, 2))

In [ ]:
u = copula.pobs(ro.FloatVector(residuals_standardized_matrix)) # Get the pseudo-observations from the standardized residuals
u

NotImplementedError: Conversion 'py2rpy' not defined for objects of type '<class 'numpy.ndarray'>'

In [56]:
u2.shape[1]

2

#### Copulas

In [60]:
# Gaussian copula
cop_gaussian = copula.normalCopula(dim=u2.shape[1], dispstr="un")

fit_gaussian = copula.fitCopula(cop_gaussian, u2, method = "mpl") # Fit the Gaussian copula to the pseudo-observations

NotImplementedError: Conversion 'py2rpy' not defined for objects of type '<class 'numpy.ndarray'>'